## 1. Convert Existing Notebooks to Importable Modules

Extracts all function definitions from both notebooks into `.py` modules automatically.

In [ ]:
# ============================================================================
# NOTE: graph_features_lib.py and stat_features_lib.py are proper modules
# (pure function definitions only — no execution code at import time).
# This cell is kept for reference. Do NOT re-run auto-extraction; edit the
# library files directly.
# ============================================================================
print("Library files are pre-built. Proceed to Cell 3 (Imports).")
print("  graph_features_lib.py — all graph + topology + IncrementalTopology functions")
print("  stat_features_lib.py  — statistical feature functions")


## 2. Imports

In [ ]:
import os
import sys
import json
import time
import logging
import warnings
import numpy as np
import pandas as pd
import networkx as nx
import bgpkit
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import List, Tuple, Dict, Set
from collections import defaultdict, Counter
from urllib.request import urlretrieve
from urllib.error import URLError, HTTPError
from tqdm.auto import tqdm

try:
    import networkit as nk
    HAS_NETWORKIT = True
except ImportError:
    HAS_NETWORKIT = False

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s: %(message)s')
logger = logging.getLogger('bgp_unified')

print(f'Pandas {pd.__version__}, NumPy {np.__version__}, NetworkX {nx.__version__}')
print(f'NetworKit: {"available" if HAS_NETWORKIT else "NOT available"}')

## 3. Configuration

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# --- Pipeline mode ---
# 'graph_only' -> extract graph + node views only
# 'stat_only'  -> extract statistical view only, still filtered by the dynamic
#                 ego scope around TARGET_AS
# 'combined'   -> extract graph + node + statistical views and merge them
PIPELINE_MODE = 'combined'
VALID_PIPELINE_MODES = {'graph_only', 'stat_only', 'combined'}
if PIPELINE_MODE not in VALID_PIPELINE_MODES:
    raise ValueError(f"Invalid PIPELINE_MODE: {PIPELINE_MODE}. Use one of {sorted(VALID_PIPELINE_MODES)}")

EXTRACT_GRAPH_VIEW = PIPELINE_MODE in {'graph_only', 'combined'}
EXTRACT_STAT_VIEW = PIPELINE_MODE in {'stat_only', 'combined'}
EXTRACT_COMBINED_VIEW = PIPELINE_MODE == 'combined'

# --- Data source ---
COLLECTOR = 'rrc04'
START_DATE = '2025-11-07'
END_DATE = '2025-11-08'   # exclusive upper boundary; processes windows before 00:00 UTC

# --- Ego-network ---
TARGET_AS = 3352
EGO_K_HOP = 2

# --- Time window ---
WINDOW_SIZE = '5min'

# --- Runtime safety ---
RUN_SANITY_CHECK = True
RESET_PREFIX_STATE_ON_SEGMENT_GAP = True
AUTO_DOWNLOAD_MRT = True

# --- Topology HTML export ---
GENERATE_TOPOLOGY_HTML = True
TOPOLOGY_HTML_MAX_NODES = 1500   # pruned HTML is created only above this size

# --- AS-path filtering ---
REMOVE_IXP_ASNS = False
IXP_ASNS = set()
RARE_AS_THRESHOLD = 3

# --- Optional enrichments ---
USE_CAIDA_RELATIONSHIPS = True
USE_IXP_FEATURES = True

# --- Graph extraction performance ---
BETWEENNESS_SAMPLE_K = None
COMPUTE_SPECTRAL = True
MAX_NODES_FOR_CLIQUE = 5000

# --- Project paths ---
PROJECT_ROOT = Path('/home/smotaali/First_Full_Paper')
BASE_DIR = PROJECT_ROOT / 'bgp_unified_results'
OUTPUT_DIR = BASE_DIR / 'output'
MRT_DIR = BASE_DIR / 'mrt_files'
SHARED_MRT_DIR = PROJECT_ROOT / 'shared_cache' / 'mrt_files'
TEMP_DIR = BASE_DIR / 'temp'
CAIDA_DIR = BASE_DIR / 'reference_data' / 'caida'

GRAPH_RESULTS_DIR = PROJECT_ROOT / 'bgp_graph_features_results'
STAT_RESULTS_DIR = PROJECT_ROOT / 'bgp_stat_features_results'

for d in [OUTPUT_DIR, MRT_DIR, SHARED_MRT_DIR, TEMP_DIR, CAIDA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SUFFIX = f"{COLLECTOR}_AS{TARGET_AS}_{EGO_K_HOP}hop_{START_DATE}_{END_DATE}_{WINDOW_SIZE}"
MODE_TAG = f"{PIPELINE_MODE}_{SUFFIX}"

CONFIG = {
    'target_as': TARGET_AS,
    'k_hop': EGO_K_HOP,
    'mode': 'ego',
    'compute_spectral': COMPUTE_SPECTRAL,
    'betweenness_sample_k': BETWEENNESS_SAMPLE_K,
    'max_nodes_for_clique': MAX_NODES_FOR_CLIQUE,
}

PIPELINE_RUN_STARTED_AT = datetime.now(timezone.utc)
PIPELINE_RUN_CLOCK_STARTED = time.perf_counter()
pipeline_timings = defaultdict(float)
exported_files = []
report = None
report_paths = []
topology_html_snapshots = []
topology_html_exports = []


def add_timing(name, elapsed):
    pipeline_timings[name] += max(0.0, float(elapsed))


print(f"Pipeline mode:     {PIPELINE_MODE}")
print(f"Collector:         {COLLECTOR}")
print(f"Date range:        {START_DATE} to {END_DATE} (END_DATE exclusive)")
print(f"Target AS:         {TARGET_AS}")
print(f"Ego hops:          {EGO_K_HOP}")
print(f"Window size:       {WINDOW_SIZE}")
print(f"Sanity check:      {'enabled' if RUN_SANITY_CHECK else 'disabled'}")
print(f"MRT auto-download: {'enabled' if AUTO_DOWNLOAD_MRT else 'disabled'}")
print(
    'Prefix carry-over: '
    f"{'reset on segment gaps' if RESET_PREFIX_STATE_ON_SEGMENT_GAP else 'always carry forward'}"
)
print(f"Graph view:        {'enabled' if EXTRACT_GRAPH_VIEW else 'disabled'}")
print(f"Stat view:         {'enabled' if EXTRACT_STAT_VIEW else 'disabled'}")
print(f"Combined output:   {'enabled' if EXTRACT_COMBINED_VIEW else 'disabled'}")
print(f"Topology HTML:     {'enabled' if GENERATE_TOPOLOGY_HTML and EXTRACT_GRAPH_VIEW else 'disabled'}")
print(f"CAIDA features:    {'enabled' if USE_CAIDA_RELATIONSHIPS and EXTRACT_GRAPH_VIEW else 'disabled'}")
print(f"IXP features:      {'enabled' if USE_IXP_FEATURES and EXTRACT_GRAPH_VIEW else 'disabled'}")
print(f"Output:            {OUTPUT_DIR}")
print(f"Shared MRT cache:  {SHARED_MRT_DIR}")



## 4. Import Functions from Existing Notebooks

Load the auto-extracted modules. These contain all functions from both notebooks:
- `graph_features_lib`: `parse_as_path`, `extract_edges_from_as_path`, `extract_graph_level_features`, `extract_node_level_features`, `extract_relationship_features`, `nx_to_nk`, `extract_ego_subgraph`, etc.
- `stat_features_lib`: `calculate_edit_distance`, `extract_statistical_features`, etc.

**Note:** If auto-import fails, copy the function cells manually from both notebooks.

In [ ]:
# ============================================================================
# Import all functions from the proper library modules.
# These files contain ONLY function/class definitions — no execution code.
# ============================================================================

import sys
from pathlib import Path

# Ensure the Scripts directory is on the path
_scripts_dir = str(Path(__file__).parent) if '__file__' in dir() else '.'
if _scripts_dir not in sys.path:
    sys.path.insert(0, _scripts_dir)

import importlib

# ── Graph features library ──
try:
    import graph_features_lib
    importlib.reload(graph_features_lib)
    from graph_features_lib import (
        # ASN utilities
        is_valid_public_asn, parse_as_path, parse_as_path_to_str, extract_edges_from_as_path,
        # MRT parsing
        download_mrt_file, parse_mrt_to_rows, build_edges_from_csv,
        # CAIDA
        find_caida_date, download_caida_rel2, load_caida_relationships,
        # Graph utilities
        nx_to_nk, extract_ego_subgraph,
        # Feature extraction
        extract_graph_level_features, extract_node_level_features,
        extract_relationship_features,
        # IXP features
        download_peeringdb_dump, load_peeringdb_ixp_memberships,
        build_ixp_feature_vectors, compute_ixp_cosine_features,
        # Incremental topology (moved here from separate cell)
        IncrementalTopology,
    )
    print('graph_features_lib: imported ✓')
    print(f'  HAS_NETWORKIT = {graph_features_lib.HAS_NETWORKIT}')
except Exception as e:
    print(f'graph_features_lib: IMPORT FAILED — {e}')
    raise

# ── Statistical features library ──
try:
    import stat_features_lib
    importlib.reload(stat_features_lib)
    from stat_features_lib import (
        parse_as_path_clean,       # unified AS-path cleaner (private ASN filtering)
        calculate_edit_distance,
        extract_statistical_features,
        KEEP_STATISTICAL, DROP_STATISTICAL,
        download_mrt_files,
    )
    print('stat_features_lib: imported ✓')
    print('  parse_as_path_clean — filters private ASNs, removes AS-SETs, dedupes prepending')
    print('  extract_statistical_features(df, prev_prefixes, rare_as_threshold=3)')
except Exception as e:
    print(f'stat_features_lib: IMPORT FAILED — {e}')
    raise

# ── Interactive topology visualization helper ──
try:
    import topology_viz_lib
    importlib.reload(topology_viz_lib)
    from topology_viz_lib import (
        export_topology_visualizations,
        DEFAULT_MAX_VIZ_NODES,
        DEFAULT_STAT_FEATURES,
    )
    print('topology_viz_lib: imported ✓')
except Exception as e:
    print(f'topology_viz_lib: IMPORT FAILED — {e}')
    raise

# ── Propagate IXP filter settings to graph lib if configured ──
if REMOVE_IXP_ASNS:
    graph_features_lib.REMOVE_IXP_ASNS = True
    graph_features_lib.IXP_ASNS = IXP_ASNS
    print(f'  IXP filter: ENABLED ({len(IXP_ASNS)} IXP ASNs)')

if TOPOLOGY_HTML_MAX_NODES != DEFAULT_MAX_VIZ_NODES:
    print(f'  Topology HTML prune threshold: {TOPOLOGY_HTML_MAX_NODES:,} nodes')

job_checkpoint = globals().get('job_checkpoint', lambda note='': None)



In [ ]:
# ============================================================================
# Candidate input directories
# ============================================================================

SHARED_COLLECTOR_MRT_DIR = SHARED_MRT_DIR / COLLECTOR
SHARED_COLLECTOR_MRT_DIR.mkdir(parents=True, exist_ok=True)

DOWNLOAD_MRT_DIR = SHARED_COLLECTOR_MRT_DIR

# Important: MRT filenames do not encode the collector name, so only use
# collector-scoped directories here. Generic mrt_files/ roots can silently mix
# data from different collectors.
GRAPH_MRT_CANDIDATES = [
    SHARED_COLLECTOR_MRT_DIR,
    MRT_DIR / COLLECTOR,
    GRAPH_RESULTS_DIR / 'data' / 'mrt_files' / COLLECTOR,
]

UPDATE_MRT_CANDIDATES = [
    SHARED_COLLECTOR_MRT_DIR,
    MRT_DIR / COLLECTOR,
]

print('RIB local search roots:')
for candidate in GRAPH_MRT_CANDIDATES:
    print(f"  - {candidate}")

print('\nUPDATE local search roots:')
for candidate in UPDATE_MRT_CANDIDATES:
    print(f"  - {candidate}")

print(f"\nShared MRT cache:         {SHARED_COLLECTOR_MRT_DIR}")
print(f"MRT download destination: {DOWNLOAD_MRT_DIR}")


## 5. Download RIB + UPDATE Files

Download **one** RIB snapshot (starting topology) and **all** UPDATE files for the period.

In [ ]:
# ============================================================================
# Locate reusable RIB + UPDATE files, downloading any missing MRT inputs
# ============================================================================

from datetime import datetime, timedelta
import shutil


def _resolve_local_file(filename, roots, promote_dir=None):
    promote_path = None
    if promote_dir is not None:
        promote_path = promote_dir / filename
        promote_path.parent.mkdir(parents=True, exist_ok=True)
        if promote_path.exists() and promote_path.stat().st_size > 0:
            return promote_path

    for root in roots:
        candidate = root / filename
        try:
            exists = candidate.exists() and candidate.stat().st_size > 0
        except OSError:
            exists = False
        if not exists:
            continue

        if promote_path is not None and candidate != promote_path:
            try:
                shutil.copy2(candidate, promote_path)
                return promote_path
            except OSError as exc:
                logger.warning(f"Could not promote {filename} into shared MRT cache: {exc}")
        return candidate
    return None


def _cache_source_label(path):
    if path is None:
        return 'local-cache'
    try:
        return 'shared-cache' if path.resolve().parent == SHARED_COLLECTOR_MRT_DIR.resolve() else 'local-cache'
    except OSError:
        return 'shared-cache' if path.parent == SHARED_COLLECTOR_MRT_DIR else 'local-cache'


def _rib_url(collector, ts):
    return (
        f"https://data.ris.ripe.net/{collector}/{ts:%Y.%m}/"
        f"bview.{ts:%Y%m%d.%H%M}.gz"
    )


def _update_spec(collector, ts):
    filename = f"updates.{ts:%Y%m%d.%H%M}.gz"
    url = f"https://data.ris.ripe.net/{collector}/{ts:%Y.%m}/{filename}"
    return url, filename, ts


job_checkpoint('Resolving MRT inputs')
resolve_started = time.perf_counter()

start_dt = datetime.strptime(START_DATE, '%Y-%m-%d')
end_dt = datetime.strptime(END_DATE, '%Y-%m-%d')

rib_times = []
current = start_dt
while current < end_dt:
    rib_times.append(current)
    current += timedelta(hours=8)

segments = []
for rib_idx, rib_time in enumerate(rib_times):
    job_checkpoint(f"Resolving MRT segment {rib_idx + 1}/{len(rib_times)}")
    next_rib = rib_times[rib_idx + 1] if rib_idx + 1 < len(rib_times) else end_dt
    rib_fname = f"bview.{rib_time.strftime('%Y%m%d.%H%M')}.gz"

    rib_path = _resolve_local_file(
        rib_fname,
        GRAPH_MRT_CANDIDATES,
        promote_dir=SHARED_COLLECTOR_MRT_DIR,
    )
    rib_source = _cache_source_label(rib_path)
    rib_download_elapsed = 0.0
    if rib_path is None and AUTO_DOWNLOAD_MRT:
        rib_url = _rib_url(COLLECTOR, rib_time)
        try:
            job_checkpoint(f"Downloading RIB {rib_fname}")
            rib_download_started = time.perf_counter()
            rib_path = download_mrt_file(rib_url, SHARED_MRT_DIR, collector=COLLECTOR)
            rib_download_elapsed = time.perf_counter() - rib_download_started
            add_timing('download_rib', rib_download_elapsed)
            rib_source = 'downloaded'
        except Exception as e:
            logger.warning(f"RIB download failed: {rib_url} — skipping this segment ({e})")
            continue
    elif rib_path is None:
        logger.warning(f"RIB not found: {rib_fname} — skipping this segment")
        continue

    update_paths = []
    missing_update_specs = []
    current_update = rib_time
    while current_update < next_rib:
        update_url, update_fname, update_ts = _update_spec(COLLECTOR, current_update)
        update_path = _resolve_local_file(
            update_fname,
            UPDATE_MRT_CANDIDATES,
            promote_dir=SHARED_COLLECTOR_MRT_DIR,
        )
        if update_path is not None:
            update_paths.append((update_path, update_ts))
        else:
            missing_update_specs.append((update_url, update_fname, update_ts))
        current_update += timedelta(minutes=5)

    downloaded_update_paths = []
    update_download_elapsed = 0.0
    if missing_update_specs and AUTO_DOWNLOAD_MRT:
        job_checkpoint(f"Downloading missing UPDATE files for {rib_fname}")
        update_download_started = time.perf_counter()
        downloaded_update_paths = download_mrt_files(
            missing_update_specs,
            DOWNLOAD_MRT_DIR,
            progress_callback=job_checkpoint,
        )
        update_download_elapsed = time.perf_counter() - update_download_started
        add_timing('download_updates', update_download_elapsed)
        update_paths.extend(downloaded_update_paths)

    downloaded_update_times = {ts for _, ts in downloaded_update_paths}
    missing_after_download = [
        fname for _, fname, ts in missing_update_specs if ts not in downloaded_update_times
    ]
    for fname in missing_after_download[:5]:
        logger.warning(f"UPDATE not found after local search/download: {fname}")
    if len(missing_after_download) > 5:
        logger.warning(f"... {len(missing_after_download) - 5} more UPDATE files missing in this segment")

    update_paths = sorted(update_paths, key=lambda item: item[1])
    local_update_count = len(update_paths) - len(downloaded_update_paths)

    segment_timing = {
        'segment_id': rib_idx + 1,
        'rib_source': rib_source,
        'rib_download_sec': rib_download_elapsed,
        'update_download_sec': update_download_elapsed,
        'update_files_expected': len(update_paths) + len(missing_after_download),
        'update_files_local': local_update_count,
        'update_files_downloaded': len(downloaded_update_paths),
        'update_files_missing': len(missing_after_download),
        'update_rows_parsed': 0,
        'windows_nonempty': 0,
        'graph_windows': 0,
        'stat_windows': 0,
        'html_snapshot_window': None,
        'timings_sec': defaultdict(float),
    }

    segments.append({
        'segment_id': rib_idx + 1,
        'rib_path': rib_path,
        'rib_fname': rib_fname,
        'rib_time': rib_time,
        'next_rib_time': next_rib,
        'update_paths': update_paths,
        'timing': segment_timing,
    })

    print(
        f"Segment {rib_idx + 1}: {rib_fname} [{rib_source}] | "
        f"updates ready = {len(update_paths)} "
        f"(local={local_update_count}, downloaded={len(downloaded_update_paths)}, missing={len(missing_after_download)}) | "
        f"window = {rib_time:%Y-%m-%d %H:%M} -> {next_rib:%Y-%m-%d %H:%M}"
    )

total_updates = sum(len(seg['update_paths']) for seg in segments)
print(f"\nTotal segments ready: {len(segments)}")
print(f"Total UPDATE files:   {total_updates}")

add_timing('resolve_mrt_inputs', time.perf_counter() - resolve_started)


## 6. IncrementalTopology Class

Maintains an evolving AS-level topology using reference counting.
Each edge tracks how many (peer, prefix) paths support it.
An edge is removed only when its reference count reaches zero.

**Based on:** Sanchez et al. (BigDAMA 2019), confirmed by author correspondence.

In [ ]:
# ============================================================================
# IncrementalTopology is now imported from graph_features_lib (Cell 7).
# This cell is intentionally empty — kept as a placeholder so cell numbering
# matches the markdown header above it.
# ============================================================================
print("IncrementalTopology imported from graph_features_lib ✓")
print("  Methods: load_rib(), apply_announcement(), apply_withdrawal(),")
print("           get_ego_subgraph(), snapshot_info()")


## 7. Parse All UPDATE Files into DataFrame

Parse all UPDATE files into a single DataFrame for windowed processing.

In [ ]:
# ============================================================================
# Verify located files
# ============================================================================

total_updates = sum(len(seg['update_paths']) for seg in segments)
total_ribs = len(segments)

print(f"Segments:      {total_ribs}")
print(f"Update files:  {total_updates} total across all segments")
for seg in segments:
    print(
        f"  Segment {seg['segment_id']}: "
        f"RIB {seg['rib_time']:%Y-%m-%d %H:%M} -> {seg['next_rib_time']:%Y-%m-%d %H:%M}, "
        f"{len(seg['update_paths'])} update files"
    )
    if not seg['rib_path'].exists():
        print(f"    WARNING: missing RIB file: {seg['rib_path']}")

print('\nAll local/downloaded files resolved. Run Cell 16 for the topology sanity check, then Cell 17/19.')


## 8. Initialize Topology from RIB + Load CAIDA Relationships

In [ ]:
# ============================================================================
# Quick sanity-check: load the FIRST segment's RIB into a temporary topology
# to verify AS{TARGET_AS} is reachable before committing to the full run.
# The main loop (Cell 19) resets the topology fresh for each segment.
# The site runner disables this to avoid an extra RIB parse per job.
# ============================================================================

if not segments:
    raise RuntimeError("No segments found. Check Cell 10 (file discovery).")

if not RUN_SANITY_CHECK:
    print("Sanity check skipped (RUN_SANITY_CHECK=False).")
else:
    first_rib = segments[0]['rib_path']   # always use segments[0], not the loop variable
    topo = IncrementalTopology()
    topo.load_rib(str(first_rib))

    info = topo.snapshot_info()
    print(f"\nInitial topology (first segment RIB):")
    print(f"  Nodes: {info['n_nodes']:,}")
    print(f"  Edges: {info['n_edges']:,}")
    print(f"  Tracked (peer,prefix) paths: {info['n_tracked_paths']:,}")

    target_int = int(TARGET_AS)
    if target_int not in topo.G:
        raise AssertionError(f"AS{TARGET_AS} not found in initial topology — check TARGET_AS config.")

    print(f"  AS{TARGET_AS} degree: {topo.G.degree(target_int)}")

    # Test ego subgraph
    test_ego = topo.get_ego_subgraph(TARGET_AS, EGO_K_HOP)
    print(f"  Initial ego ({EGO_K_HOP}-hop): {test_ego.number_of_nodes()} nodes, "
          f"{test_ego.number_of_edges()} edges")
    print(f"\nSanity check passed. AS{TARGET_AS} is present and reachable.")


In [ ]:
# ============================================================================
# Load CAIDA relationship data and PeeringDB IXP memberships
# ============================================================================

rel_map = {}
caida_meta = {'clique': set(), 'ixp_ases': set(), 'sources': []}
HAS_CAIDA = False

if EXTRACT_GRAPH_VIEW and USE_CAIDA_RELATIONSHIPS:
    caida_started = time.perf_counter()
    try:
        caida_date = find_caida_date(START_DATE)
        caida_path = download_caida_rel2(caida_date, CAIDA_DIR)
        rel_map, caida_meta = load_caida_relationships(caida_path)
        HAS_CAIDA = True
        print(f"CAIDA loaded: {caida_path.name} ({len(rel_map):,} directed relationships)")

        if REMOVE_IXP_ASNS and caida_meta.get('ixp_ases'):
            graph_features_lib.IXP_ASNS = set(caida_meta['ixp_ases']) | set(IXP_ASNS)
            print(f"  IXP ASN filter populated from CAIDA: {len(graph_features_lib.IXP_ASNS):,} ASNs")
    except Exception as e:
        logger.warning(f"CAIDA load failed: {e}. Relationship features will be skipped.")
    finally:
        add_timing('load_caida_relationships', time.perf_counter() - caida_started)
else:
    print('CAIDA load skipped for this mode.')

ixp_memberships = {}
HAS_PEERINGDB = False

if EXTRACT_GRAPH_VIEW and USE_IXP_FEATURES:
    pdb_started = time.perf_counter()
    try:
        pdb_path = download_peeringdb_dump(date_str=START_DATE, dest_dir=TEMP_DIR)
        if pdb_path is None:
            raise FileNotFoundError('No PeeringDB dump found within the fallback window.')
        ixp_memberships = load_peeringdb_ixp_memberships(pdb_path)
        HAS_PEERINGDB = True
        print(f"PeeringDB loaded: {pdb_path.name} ({len(ixp_memberships):,} ASNs with memberships)")
    except Exception as e:
        logger.warning(f"PeeringDB load failed: {e}. IXP features will be skipped.")
    finally:
        add_timing('load_peeringdb', time.perf_counter() - pdb_started)
else:
    print('PeeringDB load skipped for this mode.')



## 9. Unified Processing Loop

For each 5-minute window:
1. Apply updates to evolve the topology
2. Extract ego subgraph
3. Extract **graph features** from current topology
4. Extract **statistical features** from the window's updates
5. Store both in aligned rows

In [ ]:
# ============================================================================
# MAIN LOOP: Process each segment (RIB + its UPDATE windows)
# ============================================================================

import pickle

graph_results = []
node_results = []
stat_results = []
topo_evolution = []
topology_html_snapshots = []
prev_prefixes = None
prev_stat_segment_boundary = None

CHECKPOINT_DIR = BASE_DIR / 'checkpoints'
SEGMENT_CHECKPOINT_DIR = CHECKPOINT_DIR / 'segments'
HTML_SNAPSHOT_DIR = CHECKPOINT_DIR / 'html_snapshots'
RESUME_STATE_PATH = CHECKPOINT_DIR / 'resume_state.json'
for d in [CHECKPOINT_DIR, SEGMENT_CHECKPOINT_DIR, HTML_SNAPSHOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

target_int = int(TARGET_AS)
target_str = str(TARGET_AS)

stat_feature_names = list(dict.fromkeys(KEEP_STATISTICAL + DROP_STATISTICAL))
stat_float_features = {
    'ann_rate', 'wd_rate', 'wd_ann_ratio', 'ann_wd_ratio',
    'as_path_avg', 'as_path_std', 'edit_distance_avg', 'rare_ases_ratio',
}
completed_segment_ids = set()
restored_segment_ids = set()
resumed_from_checkpoint = False


def _segment_checkpoint_path(seg_idx, kind, suffix='json'):
    return SEGMENT_CHECKPOINT_DIR / f"segment_{int(seg_idx):03d}_{kind}.{suffix}"


def _html_snapshot_path(seg_idx):
    return HTML_SNAPSHOT_DIR / f"segment_{int(seg_idx):03d}_html_snapshot.pkl"


def _write_json_atomic(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + '.tmp')
    with open(temp_path, 'w', encoding='utf-8') as handle:
        json.dump(payload, handle, indent=2, default=str)
    temp_path.replace(path)


def _write_records_json(path, records):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + '.tmp')
    pd.DataFrame(records).to_json(temp_path, orient='records', date_format='iso')
    temp_path.replace(path)


def _write_pickle_atomic(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp_path = path.with_suffix(path.suffix + '.tmp')
    with open(temp_path, 'wb') as handle:
        pickle.dump(payload, handle, protocol=pickle.HIGHEST_PROTOCOL)
    temp_path.replace(path)


def _read_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return {} if default is None else default
    try:
        with open(path, 'r', encoding='utf-8') as handle:
            return json.load(handle)
    except Exception:
        return {} if default is None else default


def _read_records_json(path):
    path = Path(path)
    if not path.exists():
        return []
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        return []
    return payload if isinstance(payload, list) else []


def _read_pickle(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    try:
        with open(path, 'rb') as handle:
            return pickle.load(handle)
    except Exception:
        return default


def _coerce_resume_dt(value):
    if not value:
        return None
    dt = datetime.fromisoformat(value)
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)
    return dt


def _get_origin_as(path_str):
    if isinstance(path_str, str) and path_str:
        parts = path_str.split()
        return parts[-1] if parts else ''
    return ''


def _empty_stat_features():
    return {
        name: (0.0 if name in stat_float_features else 0)
        for name in stat_feature_names
    }


def _add_segment_timing(seg, name, elapsed):
    elapsed = max(0.0, float(elapsed))
    seg['timing']['timings_sec'][name] += elapsed
    add_timing(name, elapsed)


def _stat_features_for_html(stat_feats):
    return {
        feature: stat_feats.get(feature)
        for feature in DEFAULT_STAT_FEATURES
        if feature in stat_feats
    }


def _segment_timing_payload(seg):
    timing = seg.get('timing', {})
    return {
        'segment_id': seg['segment_id'],
        'rib_time': seg['rib_time'].isoformat(),
        'next_rib_time': seg['next_rib_time'].isoformat(),
        'rib_source': timing.get('rib_source'),
        'update_files_expected': int(timing.get('update_files_expected', 0)),
        'update_files_ready': int(len(seg.get('update_paths', []))),
        'update_files_local': int(timing.get('update_files_local', 0)),
        'update_files_downloaded': int(timing.get('update_files_downloaded', 0)),
        'update_files_missing': int(timing.get('update_files_missing', 0)),
        'update_rows_parsed': int(timing.get('update_rows_parsed', 0)),
        'windows_nonempty': int(timing.get('windows_nonempty', 0)),
        'graph_windows': int(timing.get('graph_windows', 0)),
        'stat_windows': int(timing.get('stat_windows', 0)),
        'html_snapshot_window': timing.get('html_snapshot_window'),
        'timings_sec': {
            key: round(float(value), 6)
            for key, value in sorted(timing.get('timings_sec', {}).items())
            if float(value) > 0
        },
    }


def _restore_segment_timing(seg, payload):
    if not payload:
        return
    timing = seg['timing']
    for key in [
        'rib_source',
        'update_files_expected',
        'update_files_local',
        'update_files_downloaded',
        'update_files_missing',
        'update_rows_parsed',
        'windows_nonempty',
        'graph_windows',
        'stat_windows',
        'html_snapshot_window',
    ]:
        if key in payload:
            timing[key] = payload[key]
    restored_timings = defaultdict(float)
    for key, value in payload.get('timings_sec', {}).items():
        restored_timings[key] += float(value)
    timing['timings_sec'] = restored_timings


def _persist_segment_checkpoint(
    seg,
    *,
    graph_rows,
    node_rows,
    stat_rows,
    topo_rows,
    html_snapshot,
    prev_prefixes_after,
    prev_boundary_after,
):
    seg_idx = seg['segment_id']
    _write_records_json(_segment_checkpoint_path(seg_idx, 'graph'), graph_rows)
    _write_records_json(_segment_checkpoint_path(seg_idx, 'node'), node_rows)
    _write_records_json(_segment_checkpoint_path(seg_idx, 'stat'), stat_rows)
    _write_records_json(_segment_checkpoint_path(seg_idx, 'topology'), topo_rows)
    _write_json_atomic(_segment_checkpoint_path(seg_idx, 'timing'), _segment_timing_payload(seg))
    _write_json_atomic(
        _segment_checkpoint_path(seg_idx, 'state'),
        {
            'segment_id': seg_idx,
            'prev_prefixes_after': sorted(prev_prefixes_after) if prev_prefixes_after is not None else None,
            'prev_stat_segment_boundary_after': (
                prev_boundary_after.isoformat() if prev_boundary_after is not None else None
            ),
        },
    )

    snapshot_path = _html_snapshot_path(seg_idx)
    if html_snapshot is not None:
        _write_pickle_atomic(snapshot_path, html_snapshot)
    elif snapshot_path.exists():
        snapshot_path.unlink(missing_ok=True)


def _save_resume_state():
    _write_json_atomic(
        RESUME_STATE_PATH,
        {
            'format_version': 1,
            'completed_segments': sorted(completed_segment_ids),
            'last_completed_segment': max(completed_segment_ids) if completed_segment_ids else 0,
            'pipeline_timings': {
                key: round(float(value), 6)
                for key, value in sorted(pipeline_timings.items())
                if float(value) > 0
            },
            'run_started_at': PIPELINE_RUN_STARTED_AT.isoformat(),
        },
    )


current_session_timings = defaultdict(float)
for key, value in pipeline_timings.items():
    current_session_timings[key] += float(value)

resume_state = _read_json(RESUME_STATE_PATH, default={})
if resume_state.get('run_started_at'):
    PIPELINE_RUN_STARTED_AT = _coerce_resume_dt(resume_state['run_started_at']) or PIPELINE_RUN_STARTED_AT

restored_pipeline_timings = defaultdict(float)
for key, value in resume_state.get('pipeline_timings', {}).items():
    restored_pipeline_timings[key] += float(value)

segment_lookup = {seg['segment_id']: seg for seg in segments}
for timing_path in sorted(SEGMENT_CHECKPOINT_DIR.glob('segment_*_timing.json')):
    parts = timing_path.stem.split('_')
    if len(parts) < 3:
        continue
    try:
        seg_id = int(parts[1])
    except ValueError:
        continue
    if seg_id in segment_lookup:
        completed_segment_ids.add(seg_id)

for seg_id in sorted(int(item) for item in resume_state.get('completed_segments', [])):
    if seg_id in segment_lookup:
        completed_segment_ids.add(seg_id)

segment_timing_totals = defaultdict(float)
for seg_id in sorted(completed_segment_ids):
    restored_segment_ids.add(seg_id)

    graph_results.extend(_read_records_json(_segment_checkpoint_path(seg_id, 'graph')))
    node_results.extend(_read_records_json(_segment_checkpoint_path(seg_id, 'node')))
    stat_results.extend(_read_records_json(_segment_checkpoint_path(seg_id, 'stat')))
    topo_evolution.extend(_read_records_json(_segment_checkpoint_path(seg_id, 'topology')))

    snapshot_payload = _read_pickle(_html_snapshot_path(seg_id))
    if snapshot_payload is not None:
        topology_html_snapshots.append(snapshot_payload)

    timing_payload = _read_json(_segment_checkpoint_path(seg_id, 'timing'), default={})
    if timing_payload:
        _restore_segment_timing(segment_lookup[seg_id], timing_payload)
        for key, value in timing_payload.get('timings_sec', {}).items():
            segment_timing_totals[key] += float(value)

for key, value in segment_timing_totals.items():
    restored_pipeline_timings[key] = float(value)

pipeline_timings.clear()
for key, value in restored_pipeline_timings.items():
    pipeline_timings[key] += float(value)
for key, value in current_session_timings.items():
    pipeline_timings[key] += float(value)

if completed_segment_ids:
    resumed_from_checkpoint = True
    last_completed_segment = max(completed_segment_ids)
    state_payload = _read_json(_segment_checkpoint_path(last_completed_segment, 'state'), default={})
    prefix_state = state_payload.get('prev_prefixes_after')
    prev_prefixes = set(prefix_state) if prefix_state is not None else None
    prev_stat_segment_boundary = _coerce_resume_dt(state_payload.get('prev_stat_segment_boundary_after'))

    print(f"\n{'=' * 70}")
    print('RESUMING FROM SEGMENT CHECKPOINTS')
    print(f"{'=' * 70}")
    print(f"Completed segments restored: {len(completed_segment_ids)} / {len(segments)}")
    print(f"Last completed segment:     {last_completed_segment}")
    print(f"Restored graph rows:       {len(graph_results)}")
    print(f"Restored node rows:        {len(node_results)}")
    print(f"Restored stat rows:        {len(stat_results)}")
    print(f"Restored topo rows:        {len(topo_evolution)}")
    print(f"Restored HTML snapshots:   {len(topology_html_snapshots)}")
    print(f"{'=' * 70}")
    job_checkpoint(
        f"Recovered {len(completed_segment_ids)} completed segments from checkpoint; resuming remaining work"
    )

for seg in segments:
    seg_idx = seg['segment_id']
    if seg_idx in completed_segment_ids:
        print(f"  Segment {seg_idx}/{len(segments)} restored from checkpoint — skipping recomputation")
        continue

    seg_timer_started = time.perf_counter()
    seg_html_captured = False
    seg_graph_rows = []
    seg_node_rows = []
    seg_stat_rows = []
    seg_topo_rows = []
    seg_html_snapshot = None
    seg_prev_prefixes = set(prev_prefixes) if prev_prefixes is not None else None
    job_checkpoint(f"Processing segment {seg_idx}/{len(segments)}")
    print(f"\n{'=' * 70}")
    print(
        f"SEGMENT {seg_idx}/{len(segments)}: "
        f"RIB {seg['rib_time']:%Y-%m-%d %H:%M} -> {seg['next_rib_time']:%Y-%m-%d %H:%M}"
    )
    print(f"{'=' * 70}")

    if (
        EXTRACT_STAT_VIEW
        and RESET_PREFIX_STATE_ON_SEGMENT_GAP
        and seg_prev_prefixes is not None
        and prev_stat_segment_boundary is not None
        and seg['rib_time'] > prev_stat_segment_boundary
    ):
        logger.info(
            '  Resetting prev_prefixes due to a non-contiguous segment gap: '
            f"{prev_stat_segment_boundary:%Y-%m-%d %H:%M} -> {seg['rib_time']:%Y-%m-%d %H:%M}"
        )
        seg_prev_prefixes = None

    rib_load_started = time.perf_counter()
    topo = IncrementalTopology()
    topo.load_rib(str(seg['rib_path']))
    _add_segment_timing(seg, 'load_rib_topology', time.perf_counter() - rib_load_started)
    info = topo.snapshot_info()
    print(f"  Initial topology: {info['n_nodes']:,} nodes, {info['n_edges']:,} edges")

    if target_int not in topo.G:
        logger.warning(f"  AS{TARGET_AS} not present in the segment-start topology — skipping segment")
        _add_segment_timing(seg, 'segment_total', time.perf_counter() - seg_timer_started)
        _persist_segment_checkpoint(
            seg,
            graph_rows=seg_graph_rows,
            node_rows=seg_node_rows,
            stat_rows=seg_stat_rows,
            topo_rows=seg_topo_rows,
            html_snapshot=seg_html_snapshot,
            prev_prefixes_after=seg_prev_prefixes,
            prev_boundary_after=prev_stat_segment_boundary,
        )
        completed_segment_ids.add(seg_idx)
        _save_resume_state()
        continue

    seg_records = {
        'timestamp': [],
        'type': [],
        'peer_ip': [],
        'peer_as': [],
        'prefix': [],
        'as_path': [],
        'origin': [],
        'next_hop': [],
        'local_pref': [],
        'med': [],
        'community': [],
    }
    seg_record_timestamps = seg_records['timestamp']
    seg_record_types = seg_records['type']
    seg_record_peer_ips = seg_records['peer_ip']
    seg_record_peer_as = seg_records['peer_as']
    seg_record_prefixes = seg_records['prefix']
    seg_record_paths = seg_records['as_path']
    seg_record_origins = seg_records['origin']
    seg_record_next_hops = seg_records['next_hop']
    seg_record_local_prefs = seg_records['local_pref']
    seg_record_meds = seg_records['med']
    seg_record_communities = seg_records['community']
    parse_started = time.perf_counter()
    for gz_path, _ in tqdm(seg['update_paths'], desc=f'  Parsing seg {seg_idx} updates'):
        job_checkpoint(f"Parsing UPDATE file {gz_path.name}")
        try:
            parser = bgpkit.Parser(url=str(gz_path))
            for elem in parser:
                seg_record_timestamps.append(datetime.fromtimestamp(elem.timestamp, tz=timezone.utc))
                seg_record_types.append(elem.elem_type)
                seg_record_peer_ips.append(elem.peer_ip or '')
                seg_record_peer_as.append(str(elem.peer_asn) if elem.peer_asn else '')
                seg_record_prefixes.append(elem.prefix or '')
                seg_record_paths.append(elem.as_path or '')
                seg_record_origins.append(elem.origin or '')
                seg_record_next_hops.append(elem.next_hop or '')
                seg_record_local_prefs.append(elem.local_pref if elem.local_pref is not None else '')
                seg_record_meds.append(elem.med if elem.med is not None else '')
                seg_record_communities.append(' '.join(str(c) for c in elem.communities) if elem.communities else '')
        except Exception as e:
            logger.warning(f"  Error parsing {gz_path.name}: {e}")
    _add_segment_timing(seg, 'parse_updates', time.perf_counter() - parse_started)

    if not seg_record_timestamps:
        logger.warning('  No UPDATE records parsed — skipping segment')
        _add_segment_timing(seg, 'segment_total', time.perf_counter() - seg_timer_started)
        _persist_segment_checkpoint(
            seg,
            graph_rows=seg_graph_rows,
            node_rows=seg_node_rows,
            stat_rows=seg_stat_rows,
            topo_rows=seg_topo_rows,
            html_snapshot=seg_html_snapshot,
            prev_prefixes_after=seg_prev_prefixes,
            prev_boundary_after=prev_stat_segment_boundary,
        )
        completed_segment_ids.add(seg_idx)
        _save_resume_state()
        continue

    seg['timing']['update_rows_parsed'] = len(seg_record_timestamps)
    seg_df = pd.DataFrame(seg_records)
    seg_df['timestamp'] = pd.to_datetime(seg_df['timestamp'], utc=True)
    seg_df = seg_df.sort_values('timestamp').reset_index(drop=True)
    seg_df['as_path_clean'] = seg_df['as_path'].apply(parse_as_path_clean)

    print(
        f"  Parsed UPDATE rows: {len(seg_df):,} "
        f"(A={(seg_df['type'] == 'A').sum():,}, W={(seg_df['type'] == 'W').sum():,})"
    )

    grouped = seg_df.set_index('timestamp').groupby(pd.Grouper(freq=WINDOW_SIZE))
    seg_graph_count = 0
    seg_stat_count = 0
    seg_window_count = 0

    for window_start, window_df in grouped:
        if window_df.empty:
            continue

        seg_window_count += 1
        window_df_reset = window_df.reset_index()
        window_id = window_start.strftime('%Y%m%d_%H%M')
        job_checkpoint(f"Processing window {window_id}")

        apply_started = time.perf_counter()
        topo.apply_window_updates(
            window_df_reset[['type', 'peer_as', 'prefix', 'as_path']].itertuples(index=False, name=None)
        )
        _add_segment_timing(seg, 'apply_updates', time.perf_counter() - apply_started)

        topo_info = topo.snapshot_info()
        seg_topo_rows.append({
            'window_start': window_start,
            'window_id': window_id,
            'collector': COLLECTOR,
            'segment': seg_idx,
            'n_nodes': topo_info['n_nodes'],
            'n_edges': topo_info['n_edges'],
        })

        ego_G = topo.get_ego_subgraph(TARGET_AS, EGO_K_HOP)
        ego_node_count = ego_G.number_of_nodes()
        if ego_node_count == 0:
            logger.info(f"  Window {window_id}: target AS not reachable in the current topology")
            continue

        ego_asns = set(str(node) for node in ego_G.nodes())
        graph_window_ready = ego_node_count >= 3
        html_snapshot_context = None

        if EXTRACT_COMBINED_VIEW and not graph_window_ready:
            logger.info(
                f"  Window {window_id}: combined mode skipped because graph extraction needs >= 3 ego nodes "
                f"(got {ego_node_count})"
            )
            continue

        if EXTRACT_GRAPH_VIEW:
            if not graph_window_ready:
                logger.info(
                    f"  Window {window_id}: graph extraction skipped because the ego graph is too small "
                    f"({ego_node_count} nodes)"
                )
            else:
                graph_started = time.perf_counter()
                try:
                    G_lcc = ego_G if nx.is_connected(ego_G) else ego_G.subgraph(
                        max(nx.connected_components(ego_G), key=len)
                    ).copy()

                    window_announcements = window_df_reset[
                        (window_df_reset['type'] == 'A')
                        & window_df_reset['as_path_clean'].notna()
                        & (window_df_reset['as_path_clean'] != '')
                    ].copy()

                    G_nk, nx2nk_map, nk2nx_map = nx_to_nk(G_lcc)
                    graph_feats, shared_data = extract_graph_level_features(
                        G_lcc, G_nk, nk2nx_map, CONFIG
                    )
                    graph_feats.update({
                        'window_start': window_start,
                        'window_id': window_id,
                        'collector': COLLECTOR,
                        'segment': seg_idx,
                        'ego_nodes': ego_G.number_of_nodes(),
                        'ego_edges': ego_G.number_of_edges(),
                    })

                    node_df_snap, extra_graph = extract_node_level_features(
                        G_lcc, G_nk, nx2nk_map, nk2nx_map, shared_data, CONFIG
                    )
                    graph_feats.update(extra_graph)

                    if HAS_CAIDA:
                        rel_graph_feats, rel_node_df = extract_relationship_features(
                            G_lcc,
                            rel_map,
                            caida_meta,
                            target_as=target_int,
                            as_paths=window_announcements['as_path_clean'].tolist(),
                        )
                        graph_feats.update(rel_graph_feats)
                        node_df_snap = node_df_snap.join(rel_node_df, how='left')

                    if HAS_PEERINGDB:
                        ixp_vectors_df, _ = build_ixp_feature_vectors(G_lcc, ixp_memberships)
                        ixp_graph_feats, ixp_node_df = compute_ixp_cosine_features(
                            G_lcc, ixp_vectors_df, ixp_memberships
                        )
                        graph_feats.update(ixp_graph_feats)
                        node_df_snap = node_df_snap.join(ixp_node_df, how='left')

                    seg_graph_rows.append(graph_feats)

                    if target_int in node_df_snap.index:
                        target_row = node_df_snap.loc[target_int].to_dict()
                        target_row.update({
                            'asn': target_int,
                            'window_start': window_start,
                            'window_id': window_id,
                            'collector': COLLECTOR,
                            'segment': seg_idx,
                        })
                        seg_node_rows.append(target_row)
                    else:
                        logger.warning(f"  Window {window_id}: target AS missing from node-level output")

                    seg_graph_count += 1
                    seg['timing']['graph_windows'] += 1
                    if GENERATE_TOPOLOGY_HTML and not seg_html_captured:
                        html_snapshot_context = {
                            'segment_id': seg_idx,
                            'window_id': window_id,
                            'window_start': window_start,
                            'graph': G_lcc.copy(),
                            'node_df': node_df_snap.reset_index(),
                            'degeneracy': graph_feats.get('degeneracy', 0),
                        }
                except Exception as e:
                    logger.warning(f"  Graph feature extraction failed at {window_id}: {e}")
                    if EXTRACT_COMBINED_VIEW:
                        continue
                finally:
                    _add_segment_timing(seg, 'graph_extraction', time.perf_counter() - graph_started)

        if EXTRACT_STAT_VIEW:
            stat_started = time.perf_counter()
            wdf = window_df_reset.copy()
            wdf['_origin_as'] = wdf['as_path_clean'].apply(_get_origin_as)

            mask_origin = wdf['_origin_as'].isin(ego_asns)
            mask_target = wdf['as_path_clean'].str.contains(
                rf'(?:^|\s){target_str}(?:\s|$)', regex=True, na=False
            )
            mask_peer_wd = (wdf['type'] == 'W') & wdf['peer_as'].isin(ego_asns)

            ego_window_df = (
                wdf[mask_origin | mask_target | mask_peer_wd]
                .drop(columns=['_origin_as'])
                .reset_index(drop=True)
            )

            try:
                if not ego_window_df.empty:
                    stat_feats, current_prefixes = extract_statistical_features(
                        ego_window_df,
                        prev_prefixes=seg_prev_prefixes,
                        rare_as_threshold=RARE_AS_THRESHOLD,
                    )
                else:
                    stat_feats = _empty_stat_features()
                    current_prefixes = seg_prev_prefixes if seg_prev_prefixes is not None else set()

                stat_feats.update({
                    'window_start': window_start,
                    'window_id': window_id,
                    'collector': COLLECTOR,
                    'segment': seg_idx,
                    'ego_nodes_dynamic': ego_node_count,
                    'ego_updates_total': len(window_df_reset),
                    'ego_updates_filtered': len(ego_window_df),
                    'ego_filter_ratio': len(ego_window_df) / len(window_df_reset),
                })
                seg_stat_rows.append(stat_feats)
                seg_prev_prefixes = current_prefixes
                seg_stat_count += 1
                seg['timing']['stat_windows'] += 1

                if html_snapshot_context is not None and not seg_html_captured:
                    html_snapshot_context['stat_features'] = _stat_features_for_html(stat_feats)
                    seg_html_snapshot = html_snapshot_context
                    seg_html_captured = True
                    seg['timing']['html_snapshot_window'] = window_id
            except Exception as e:
                logger.warning(f"  Statistical feature extraction failed at {window_id}: {e}")
                fallback = _empty_stat_features()
                fallback.update({
                    'window_start': window_start,
                    'window_id': window_id,
                    'collector': COLLECTOR,
                    'segment': seg_idx,
                    'ego_nodes_dynamic': ego_node_count,
                    'ego_updates_total': len(window_df_reset),
                    'ego_updates_filtered': len(ego_window_df),
                    'ego_filter_ratio': len(ego_window_df) / len(window_df_reset),
                })
                seg_stat_rows.append(fallback)
                seg_stat_count += 1
                seg['timing']['stat_windows'] += 1

                if html_snapshot_context is not None and not seg_html_captured:
                    html_snapshot_context['stat_features'] = _stat_features_for_html(fallback)
                    seg_html_snapshot = html_snapshot_context
                    seg_html_captured = True
                    seg['timing']['html_snapshot_window'] = window_id
            finally:
                _add_segment_timing(seg, 'stat_extraction', time.perf_counter() - stat_started)
        elif html_snapshot_context is not None and not seg_html_captured:
            html_snapshot_context['stat_features'] = {}
            seg_html_snapshot = html_snapshot_context
            seg_html_captured = True
            seg['timing']['html_snapshot_window'] = window_id

    seg['timing']['windows_nonempty'] = seg_window_count

    if EXTRACT_STAT_VIEW and seg_stat_count > 0:
        prev_stat_segment_boundary_after = seg['next_rib_time']
    else:
        prev_stat_segment_boundary_after = prev_stat_segment_boundary

    _add_segment_timing(seg, 'segment_total', time.perf_counter() - seg_timer_started)

    graph_results.extend(seg_graph_rows)
    node_results.extend(seg_node_rows)
    stat_results.extend(seg_stat_rows)
    topo_evolution.extend(seg_topo_rows)
    if seg_html_snapshot is not None:
        topology_html_snapshots.append(seg_html_snapshot)

    prev_prefixes = seg_prev_prefixes
    prev_stat_segment_boundary = prev_stat_segment_boundary_after

    _persist_segment_checkpoint(
        seg,
        graph_rows=seg_graph_rows,
        node_rows=seg_node_rows,
        stat_rows=seg_stat_rows,
        topo_rows=seg_topo_rows,
        html_snapshot=seg_html_snapshot,
        prev_prefixes_after=prev_prefixes,
        prev_boundary_after=prev_stat_segment_boundary,
    )
    completed_segment_ids.add(seg_idx)
    _save_resume_state()

    summary_parts = []
    if EXTRACT_GRAPH_VIEW:
        summary_parts.append(f"graph rows +{seg_graph_count}")
    if EXTRACT_STAT_VIEW:
        summary_parts.append(f"stat rows +{seg_stat_count}")
    timing_parts = []
    for key in ['load_rib_topology', 'parse_updates', 'apply_updates', 'graph_extraction', 'stat_extraction']:
        value = seg['timing']['timings_sec'].get(key, 0.0)
        if value > 0:
            timing_parts.append(f"{key}={value:.2f}s")
    timing_suffix = f" | timings: {', '.join(timing_parts)}" if timing_parts else ''
    print(f"  Segment complete: {', '.join(summary_parts)}{timing_suffix}")

print(f"\n{'=' * 70}")
print('ALL SEGMENTS COMPLETE')
print(f"{'=' * 70}")
print(f"Pipeline mode:            {PIPELINE_MODE}")
if resumed_from_checkpoint:
    print(f"Recovered segments:       {len(restored_segment_ids)}")
if EXTRACT_GRAPH_VIEW:
    print(f"Graph feature rows:       {len(graph_results)}")
    print(f"Node feature rows:        {len(node_results)}")
if EXTRACT_STAT_VIEW:
    print(f"Statistical feature rows: {len(stat_results)}")
    if stat_results:
        avg_ratio = np.mean([row['ego_filter_ratio'] for row in stat_results])
        print(f"Avg ego filter ratio:     {avg_ratio:.1%} of window updates kept")
if EXTRACT_GRAPH_VIEW and GENERATE_TOPOLOGY_HTML:
    print(f"Topology HTML snapshots:  {len(topology_html_snapshots)}")
print(f"{'=' * 70}")


In [ ]:
# ============================================================================
# Post-run sanity summary
# ============================================================================

if EXTRACT_GRAPH_VIEW and not graph_results:
    raise RuntimeError('No graph windows were produced for the selected mode.')
if EXTRACT_STAT_VIEW and not stat_results:
    raise RuntimeError('No statistical windows were produced for the selected mode.')

graph_keys = sorted({key for row in graph_results for key in row.keys()}) if graph_results else []
node_keys = sorted({key for row in node_results for key in row.keys()}) if node_results else []
stat_keys = sorted({key for row in stat_results for key in row.keys()}) if stat_results else []

rel_check = ['frac_p2c_edges', 'valley_free_violation_rate', 'avg_violation_depth', 'vf_rate_delta']
ixp_graph_check = ['avg_edge_ixp_cosine_dist', 'median_edge_ixp_cosine_dist', 'std_edge_ixp_cosine_dist']
ixp_node_check = ['avg_ixp_cosine_dist', 'ixp_vector_norm', 'n_ixp_memberships']

print('Run summary:')
print(f"  Pipeline mode: {PIPELINE_MODE}")
print(f"  Graph keys:    {len(graph_keys)}")
print(f"  Node keys:     {len(node_keys)}")
print(f"  Stat keys:     {len(stat_keys)}")
if graph_keys:
    print(f"  Relationship features available: {[c for c in rel_check if c in graph_keys]}")
    print(f"  IXP graph features available:    {[c for c in ixp_graph_check if c in graph_keys]}")
if node_keys:
    print(f"  IXP node features available:     {[c for c in ixp_node_check if c in node_keys]}")
if stat_keys:
    print('  Statistical filtering uses the dynamic ego scope around TARGET_AS in every window.')


## 10. Combine into Aligned DataFrames

In [ ]:
# ============================================================================
# Build aligned DataFrames
# ============================================================================

print(f"graph_results: {len(graph_results)} rows")
print(f"node_results:  {len(node_results)} rows")
print(f"stat_results:  {len(stat_results)} rows")

graph_df = pd.DataFrame()
node_df = pd.DataFrame()
stat_df = pd.DataFrame()
combined_df = pd.DataFrame()
combined_feature_cols = []

if graph_results:
    graph_df = pd.DataFrame(graph_results)
    graph_df['window_start'] = pd.to_datetime(graph_df['window_start'])
    graph_df = graph_df.sort_values(['window_start', 'window_id']).reset_index(drop=True)

if node_results:
    node_df = pd.DataFrame(node_results)
    node_df['window_start'] = pd.to_datetime(node_df['window_start'])
    node_df = node_df.sort_values(['window_start', 'window_id']).reset_index(drop=True)

if stat_results:
    stat_df = pd.DataFrame(stat_results)
    stat_df['window_start'] = pd.to_datetime(stat_df['window_start'])
    stat_df = stat_df.sort_values(['window_start', 'window_id']).reset_index(drop=True)

print(f"\nGraph features:       {graph_df.shape}")
print(f"Node features:        {node_df.shape}")
print(f"Statistical features: {stat_df.shape}")

if EXTRACT_COMBINED_VIEW:
    if graph_df.empty or stat_df.empty:
        raise RuntimeError('Combined mode requires both graph and statistical data frames.')

    merge_keys = ['window_start', 'window_id', 'collector']
    combined_df = graph_df.copy()

    if not node_df.empty:
        node_merge_cols = [c for c in node_df.columns if c not in combined_df.columns or c in merge_keys]
        combined_df = combined_df.merge(node_df[node_merge_cols], on=merge_keys, how='left')

    stat_merge_cols = [c for c in stat_df.columns if c not in combined_df.columns or c in merge_keys]
    combined_df = combined_df.merge(stat_df[stat_merge_cols], on=merge_keys, how='left')
    combined_df = combined_df.sort_values(['window_start', 'window_id']).reset_index(drop=True)

    META_COLS = {
        'window_start', 'window_id', 'collector', 'segment', 'asn',
        'ego_nodes', 'ego_edges', 'ego_nodes_dynamic',
        'ego_updates_total', 'ego_updates_filtered', 'ego_filter_ratio'
    }
    combined_feature_cols = [c for c in combined_df.columns if c not in META_COLS and not c.startswith('_')]

    print(f"\nCOMBINED DATASET:")
    print(f"  Rows (aligned windows): {len(combined_df)}")
    print(f"  Total feature columns:  {len(combined_feature_cols)}")
    if 'vf_rate_delta' in combined_df.columns:
        print('  vf_rate_delta preserved as the CAIDA ego-vs-global valley-free delta')
else:
    print(f"\nCombined dataset not built in mode: {PIPELINE_MODE}")


## 11. Apply Feature Selection (Keep Lists from Redundancy Analysis)

In [ ]:
# ============================================================================
# Selected features from intra-view redundancy analysis
# ============================================================================

KEEP_GRAPH = [
    'n_nodes', 'n_edges', 'betweenness_std', 'betweenness_skewness',
    'rich_club_p25', 'rich_club_p95', 'algebraic_connectivity', 'spectral_gap',
    'core_mean', 'clustering_avg_local', 'avg_path_length', 'log_spanning_trees',
    'median_edge_ixp_cosine_dist', 'vf_rate_delta', 'mean_jaccard', 'mean_adamic_adar',
]

KEEP_NODE = [
    'degree', 'betweenness_centrality', 'core_number', 'eigenvector_centrality',
    'eccentricity', 'local_clustering', 'avg_neighbor_degree', 'avg_ixp_cosine_dist',
]

KEEP_REL_GRAPH = ['frac_p2c_edges', 'valley_free_violation_rate', 'avg_violation_depth']
KEEP_REL_NODE = ['n_providers', 'n_customers', 'n_peers', 'provider_ratio', 'is_tier1']

KEEP_STAT = [
    'announcements', 'withdrawals', 'wd_ann_ratio',
    'unique_prefixes_ann', 'new_prefixes', 'flaps',
    'origin_changes',
    'as_path_avg', 'as_path_max', 'unique_as_path_max',
    'edit_distance_avg', 'edit_distance_max',
    'number_rare_ases', 'imp_wd_dpath', 'unique_peers',
]

GRAPH_KEEP_ALL = KEEP_GRAPH + KEEP_REL_GRAPH
NODE_KEEP_ALL = KEEP_NODE + KEEP_REL_NODE
ALL_KEEP = KEEP_GRAPH + KEEP_NODE + KEEP_REL_GRAPH + KEEP_REL_NODE + KEEP_STAT

GRAPH_META_OUT = ['window_start', 'window_id', 'collector']
NODE_META_OUT = ['window_start', 'window_id', 'collector', 'asn']
STAT_META_OUT = ['window_start', 'window_id', 'collector']
COMBINED_META_OUT = ['window_start', 'window_id', 'collector']

def ordered_available(df, preferred_columns):
    return [column for column in preferred_columns if column in df.columns]

graph_sel = pd.DataFrame()
node_sel = pd.DataFrame()
stat_sel = pd.DataFrame()
combined_sel = pd.DataFrame()

available_graph = []
available_node = []
available_stat = []
available_combined = []
missing_combined = []

if not graph_df.empty:
    available_graph = ordered_available(graph_df, GRAPH_KEEP_ALL)
    graph_sel = graph_df[GRAPH_META_OUT + available_graph].copy()
    print(f"Graph selected features: {len(available_graph)} / {len(GRAPH_KEEP_ALL)}")

if not node_df.empty:
    available_node = ordered_available(node_df, NODE_KEEP_ALL)
    node_sel = node_df[NODE_META_OUT + available_node].copy()
    print(f"Node selected features:  {len(available_node)} / {len(NODE_KEEP_ALL)}")

if not stat_df.empty:
    available_stat = ordered_available(stat_df, KEEP_STAT)
    stat_sel = stat_df[STAT_META_OUT + available_stat].copy()
    print(f"Stat selected features:  {len(available_stat)} / {len(KEEP_STAT)}")

if EXTRACT_COMBINED_VIEW and not combined_df.empty:
    available_combined = ordered_available(combined_df, ALL_KEEP)
    missing_combined = [c for c in ALL_KEEP if c not in combined_df.columns]
    combined_sel = combined_df[COMBINED_META_OUT + available_combined].copy()

    print(f"Combined selected features: {len(available_combined)} / {len(ALL_KEEP)}")
    if missing_combined:
        print(f"  Missing combined features: {missing_combined}")


## 12. Export Results

In [ ]:
# ============================================================================
# Export
# ============================================================================

META_OUT = ['window_start', 'window_id', 'collector']


def reorder_columns(df, meta_cols):
    meta = [c for c in meta_cols if c in df.columns]
    rest = sorted([c for c in df.columns if c not in meta])
    return df[meta + rest]


def _relative_output_path(path):
    path = Path(path)
    try:
        return path.relative_to(OUTPUT_DIR).as_posix()
    except ValueError:
        return path.name


_exported_file_set = set(exported_files)


def record_export(path):
    rel_path = _relative_output_path(path)
    if rel_path not in _exported_file_set:
        exported_files.append(rel_path)
        _exported_file_set.add(rel_path)
    return rel_path


def _round_timing_map(values):
    return {
        key: round(float(value), 3)
        for key, value in sorted(values.items())
        if float(value) > 0
    }


def _serialize_segment_timing(seg):
    timing = seg.get('timing', {})
    return {
        'segment_id': seg['segment_id'],
        'rib_time': seg['rib_time'].isoformat(),
        'next_rib_time': seg['next_rib_time'].isoformat(),
        'rib_source': timing.get('rib_source'),
        'update_files_expected': int(timing.get('update_files_expected', 0)),
        'update_files_ready': int(len(seg.get('update_paths', []))),
        'update_files_local': int(timing.get('update_files_local', 0)),
        'update_files_downloaded': int(timing.get('update_files_downloaded', 0)),
        'update_files_missing': int(timing.get('update_files_missing', 0)),
        'update_rows_parsed': int(timing.get('update_rows_parsed', 0)),
        'windows_nonempty': int(timing.get('windows_nonempty', 0)),
        'graph_windows': int(timing.get('graph_windows', 0)),
        'stat_windows': int(timing.get('stat_windows', 0)),
        'html_snapshot_window': timing.get('html_snapshot_window'),
        'timings_sec': _round_timing_map(timing.get('timings_sec', {})),
    }


def write_report_files(payload):
    for report_file in report_paths:
        with open(report_file, 'w') as handle:
            json.dump(payload, handle, indent=2, default=str)


export_started = time.perf_counter()
topology_html_exports = []
topology_viz_dir = OUTPUT_DIR / f"visualization/target_{TARGET_AS}"
topology_html_error = None

if not graph_df.empty:
    graph_df_ordered = reorder_columns(graph_df, GRAPH_META_OUT)
    graph_path = OUTPUT_DIR / f"graph_features_{SUFFIX}.csv"
    graph_df_ordered.to_csv(graph_path, index=False)
    record_export(graph_path)
    print(f"Graph features:         {graph_path.name}  ({graph_df_ordered.shape})")

if not graph_sel.empty:
    graph_sel_path = OUTPUT_DIR / f"graph_selected_{SUFFIX}.csv"
    graph_sel.to_csv(graph_sel_path, index=False)
    record_export(graph_sel_path)
    print(f"Graph selected:         {graph_sel_path.name}  ({graph_sel.shape})")

if not node_df.empty:
    node_df_ordered = reorder_columns(node_df, NODE_META_OUT)
    node_path = OUTPUT_DIR / f"node_features_{SUFFIX}.csv"
    node_df_ordered.to_csv(node_path, index=False)
    record_export(node_path)
    print(f"Node features:          {node_path.name}  ({node_df_ordered.shape})")

if not node_sel.empty:
    node_sel_path = OUTPUT_DIR / f"node_selected_{SUFFIX}.csv"
    node_sel.to_csv(node_sel_path, index=False)
    record_export(node_sel_path)
    print(f"Node selected:          {node_sel_path.name}  ({node_sel.shape})")

if not stat_df.empty:
    stat_df_ordered = reorder_columns(stat_df, STAT_META_OUT)
    stat_path = OUTPUT_DIR / f"stat_features_{SUFFIX}.csv"
    stat_df_ordered.to_csv(stat_path, index=False)
    record_export(stat_path)
    print(f"Stat features:          {stat_path.name}  ({stat_df_ordered.shape})")

if not stat_sel.empty:
    stat_sel_path = OUTPUT_DIR / f"stat_selected_{SUFFIX}.csv"
    stat_sel.to_csv(stat_sel_path, index=False)
    record_export(stat_sel_path)
    print(f"Stat selected:          {stat_sel_path.name}  ({stat_sel.shape})")

if EXTRACT_COMBINED_VIEW and not combined_df.empty:
    combined_df_ordered = reorder_columns(combined_df, COMBINED_META_OUT)
    full_path = OUTPUT_DIR / f"unified_full_{SUFFIX}.csv"
    combined_df_ordered.to_csv(full_path, index=False)
    record_export(full_path)
    print(f"Unified full:           {full_path.name}  ({combined_df_ordered.shape})")

if EXTRACT_COMBINED_VIEW and not combined_sel.empty:
    sel_path = OUTPUT_DIR / f"unified_SELECTED_{SUFFIX}.csv"
    combined_sel.to_csv(sel_path, index=False)
    record_export(sel_path)
    print(f"Unified selected:       {sel_path.name}  ({combined_sel.shape})")

topo_path = OUTPUT_DIR / f"topology_evolution_{SUFFIX}.csv"
pd.DataFrame(topo_evolution).to_csv(topo_path, index=False)
record_export(topo_path)
print(f"Topology evolution:     {topo_path.name}")

report = {
    'pipeline_mode': PIPELINE_MODE,
    'collector': COLLECTOR,
    'start_date': START_DATE,
    'end_date': END_DATE,
    'target_as': TARGET_AS,
    'ego_k_hop': EGO_K_HOP,
    'window_size': WINDOW_SIZE,
    'method': 'incremental_topology with dynamic ego filtering',
    'graph_view_enabled': EXTRACT_GRAPH_VIEW,
    'stat_view_enabled': EXTRACT_STAT_VIEW,
    'combined_output_enabled': EXTRACT_COMBINED_VIEW,
    'use_caida_relationships': HAS_CAIDA,
    'use_ixp_features': HAS_PEERINGDB,
    'topology_html_enabled': bool(EXTRACT_GRAPH_VIEW and GENERATE_TOPOLOGY_HTML),
    'topology_html_requested': bool(GENERATE_TOPOLOGY_HTML),
    'topology_html_dir': None,
    'topology_html_exports': [],
    'topology_html_error': None,
    'graph_rows': int(len(graph_df)) if not graph_df.empty else 0,
    'node_rows': int(len(node_df)) if not node_df.empty else 0,
    'stat_rows': int(len(stat_df)) if not stat_df.empty else 0,
    'combined_rows': int(len(combined_df)) if not combined_df.empty else 0,
    'graph_selected_features': available_graph,
    'node_selected_features': available_node,
    'stat_selected_features': available_stat,
    'combined_selected_features': available_combined,
    'combined_missing_features': missing_combined,
    'initial_topology': topo_evolution[0] if topo_evolution else {},
    'final_topology': topo_evolution[-1] if topo_evolution else {},
    'segment_timings': [_serialize_segment_timing(seg) for seg in segments],
    'timings_sec': _round_timing_map(pipeline_timings),
    'run_started_at': PIPELINE_RUN_STARTED_AT.isoformat(),
    'resumed_from_checkpoint': bool(resumed_from_checkpoint),
    'checkpoint_segments_restored': int(len(restored_segment_ids)),
    'checkpoint_dir': str(CHECKPOINT_DIR),
    'run_finished_at': None,
    'exported_files': exported_files,
}

report_path = OUTPUT_DIR / f"pipeline_report_{MODE_TAG}.json"
report_paths = [report_path]
record_export(report_path)

legacy_report_path = None
if EXTRACT_COMBINED_VIEW:
    legacy_report_path = OUTPUT_DIR / f"unified_report_{SUFFIX}.json"
    report_paths.append(legacy_report_path)
    record_export(legacy_report_path)

report['exported_files'] = exported_files
write_report_files(report)
print(f"Report:                 {report_path.name}")
if legacy_report_path is not None:
    print(f"Legacy combined report: {legacy_report_path.name}")

if EXTRACT_GRAPH_VIEW and GENERATE_TOPOLOGY_HTML and topology_html_snapshots:
    topology_export_started = time.perf_counter()
    print(f"\n{'=' * 70}")
    print('GENERATING INTERACTIVE TOPOLOGY HTML')
    print(f"  Snapshots: {len(topology_html_snapshots)} (first graph-ready window per segment)")
    print(f"  Output dir: {topology_viz_dir}")
    print(f"{'=' * 70}")

    try:
        for snapshot_index, snapshot in enumerate(topology_html_snapshots, start=1):
            job_checkpoint(f"Generating topology HTML {snapshot['window_id']}")
            snapshot_outputs = export_topology_visualizations(
                snapshot['graph'],
                snapshot['node_df'],
                viz_dir=topology_viz_dir,
                collector=COLLECTOR,
                snapshot_id=snapshot['window_id'],
                snapshot_ts=snapshot['window_start'],
                target_as=target_int,
                degeneracy=snapshot.get('degeneracy', 0),
                stat_features=snapshot.get('stat_features', {}),
                max_nodes=TOPOLOGY_HTML_MAX_NODES,
                copy_latest=(snapshot_index == len(topology_html_snapshots)),
            )

            export_record = {
                'segment_id': snapshot['segment_id'],
                'window_id': snapshot['window_id'],
                'window_start': snapshot['window_start'],
                'files': [],
            }
            for item in snapshot_outputs:
                rel_path = record_export(item['path'])
                export_record['files'].append({
                    'kind': item['kind'],
                    'path': rel_path,
                    'size_mb': item['size_mb'],
                    'nodes': item['nodes'],
                    'links': item['links'],
                })
                print(
                    f"  [{snapshot_index}/{len(topology_html_snapshots)}] {item['path'].name} "
                    f"({item['nodes']:,} nodes, {item['links']:,} links, {item['size_mb']:.2f} MB)"
                )
                if snapshot_index == len(topology_html_snapshots):
                    latest_path = topology_viz_dir / f"topology_latest_{item['kind']}.html"
                    if latest_path.exists():
                        record_export(latest_path)
            topology_html_exports.append(export_record)

        add_timing('export_topology_html', time.perf_counter() - topology_export_started)
    except Exception as exc:
        topology_html_error = str(exc)
        logger.warning(f"Topology HTML export failed but the pipeline will continue: {exc}")

report['topology_html_dir'] = _relative_output_path(topology_viz_dir) if topology_html_exports else None
report['topology_html_exports'] = topology_html_exports
report['topology_html_error'] = topology_html_error
report['exported_files'] = exported_files
write_report_files(report)

print(f"\n{'=' * 60}")
print(f"EXPORTS WRITTEN — {OUTPUT_DIR}")
print(f"{'=' * 60}")



## 13. Visualization: Topology Evolution + Feature Time Series

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

figure_started = time.perf_counter()

topo_evo_df = pd.DataFrame(topo_evolution)
topo_evo_df['window_start'] = pd.to_datetime(topo_evo_df['window_start'])

panels = [
    {
        'label': 'Topology Size',
        'source': topo_evo_df,
        'series': [('n_nodes', 'Nodes'), ('n_edges', 'Edges')],
    }
]

if not graph_sel.empty:
    panels.append({
        'label': 'Graph Features',
        'source': graph_sel,
        'series': [('algebraic_connectivity', 'algebraic_connectivity'), ('spectral_gap', 'spectral_gap')],
    })

if not stat_sel.empty:
    panels.append({
        'label': 'Update Volume',
        'source': stat_sel,
        'series': [('announcements', 'announcements'), ('withdrawals', 'withdrawals')],
    })
    panels.append({
        'label': 'Path Dynamics',
        'source': stat_sel,
        'series': [('edit_distance_avg', 'edit_distance_avg'), ('flaps', 'flaps')],
    })

fig, axes = plt.subplots(len(panels), 1, figsize=(14, 3.5 * len(panels)), sharex=True)
if len(panels) == 1:
    axes = [axes]

for ax, panel in zip(axes, panels):
    source_df = panel['source']
    for column, label in panel['series']:
        if column in source_df.columns:
            ax.plot(source_df['window_start'], source_df[column], label=label, linewidth=0.8)
    ax.set_ylabel(panel['label'])
    ax.grid(True, alpha=0.3)
    if panel['label'] != 'Topology Size' and ax.get_legend_handles_labels()[0]:
        ax.legend(fontsize=8)

axes[0].set_title(f'BGP Pipeline Time Series — {PIPELINE_MODE} — AS{TARGET_AS} ({COLLECTOR})')
if axes[0].get_legend_handles_labels()[0]:
    axes[0].legend(fontsize=8)

axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
axes[-1].set_xlabel('Time (UTC)')
plt.tight_layout()

fig_path = OUTPUT_DIR / f"pipeline_timeseries_{MODE_TAG}.png"
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
record_export(fig_path)
print(f"Saved: {fig_path}")

if EXTRACT_COMBINED_VIEW:
    legacy_fig_path = OUTPUT_DIR / f"unified_timeseries_{SUFFIX}.png"
    plt.savefig(legacy_fig_path, dpi=150, bbox_inches='tight')
    record_export(legacy_fig_path)
    print(f"Legacy combined figure: {legacy_fig_path}")

plt.show()
plt.close(fig)

add_timing('render_timeseries_figures', time.perf_counter() - figure_started)
add_timing('export_outputs', time.perf_counter() - export_started)
report['timings_sec'] = _round_timing_map({**pipeline_timings, 'pipeline_total': time.perf_counter() - PIPELINE_RUN_CLOCK_STARTED})
report['run_finished_at'] = datetime.now(timezone.utc).isoformat()
report['exported_files'] = exported_files
write_report_files(report)
print(f"Updated reports with final timings: {report_paths[0].name}")

